In [35]:
import cv2
import numpy as np

# Cargar imagen
imagen = cv2.imread("img/IMG_20191209_100620.jpg")
imagen_gray = cv2.cvtColor(imagen, cv2.COLOR_BGR2GRAY)

# Suavizado
imagen_gray = cv2.GaussianBlur(imagen_gray, (5, 5), 0)

# Binarización
_, imagen_bin = cv2.threshold(imagen_gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

# Invertir si es necesario
# imagen_bin = cv2.bitwise_not(imagen_bin)

# Morfología
kernel = np.ones((3,3), np.uint8)
imagen_bin = cv2.morphologyEx(imagen_bin, cv2.MORPH_OPEN, kernel)

# Etiquetado
num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(imagen_bin, 8)

area_min_dado = 500  # AJUSTAR según imagen

imagen_resultado = imagen.copy()
contador_dados = 0

for i in range(1, num_labels):
    x = stats[i, cv2.CC_STAT_LEFT]
    y = stats[i, cv2.CC_STAT_TOP]
    w = stats[i, cv2.CC_STAT_WIDTH]
    h = stats[i, cv2.CC_STAT_HEIGHT]
    area = stats[i, cv2.CC_STAT_AREA]
    cX, cY = centroids[i]

    # Filtrar dados
    if area < area_min_dado or area > 5000:
        continue

    contador_dados += 1

    mascara_dado = (labels == i).astype(np.uint8) * 255

    roi_gray = imagen_gray[y:y+h, x:x+w]
    roi_mask = mascara_dado[y:y+h, x:x+w]

    # Limpiar fondo
    roi_masked = roi_gray.copy()
    roi_masked[roi_mask == 0] = 255

    # Detectar puntos
    _, puntos_bin = cv2.threshold(roi_masked, 0, 255,
                                 cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    puntos_bin = cv2.morphologyEx(puntos_bin, cv2.MORPH_OPEN,
                                 np.ones((2,2), np.uint8))

    num_puntos, labels_puntos, stats_puntos, centroids_puntos = \
        cv2.connectedComponentsWithStats(puntos_bin, 8)

    area_min_punto = 20
    puntos_validos = 0

    for j in range(1, num_puntos):
        area_punto = stats_puntos[j, cv2.CC_STAT_AREA]
        if 20 < area_punto < 500:
            puntos_validos += 1

    # CENTRAR TEXTO
    cX = x + w // 2
    cY = y + h // 2

    cv2.rectangle(imagen_resultado, (x, y), (x + w, y + h), (0, 255, 0), 2)
    cv2.circle(imagen_resultado, (int(cX), int(cY)), 4, (0, 0, 255), -1)

    cv2.putText(imagen_resultado, str(puntos_validos),
                (cX - 10, cY),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

print(f"Total de dados detectados: {contador_dados}")

# =========================
# MOSTRAR BIEN LA IMAGEN
# =========================

# Redimensionar para mejor visualización
alto, ancho = imagen_resultado.shape[:2]
escala = 800 / ancho
imagen_mostrar = cv2.resize(imagen_resultado, None, fx=escala, fy=escala)

cv2.namedWindow("Resultado", cv2.WINDOW_NORMAL)
cv2.imshow("Resultado", imagen_mostrar)
cv2.waitKey(0)
cv2.destroyAllWindows()

Total de dados detectados: 14
